# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravikiranbathe/flyrank-ai/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Ravikiranbathe/flyrank-ai.git

%cd flyrank-ai

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

fatal: destination path 'flyrank-ai' already exists and is not an empty directory.
/content/flyrank-ai
(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Rule

The baseline rule ranks pages that are likely to benefit from a content refresh. The score is based on content age, CTR, traffic trend, and impressions. Pages with higher scores are given higher priority for review and refresh. Pages with higher scores are given higher priority for review and refresh.

**Action label:**
- Refresh Content

**Reason code:**
- REFRESH_STALE

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import numpy as np
from pathlib import Path

# Create baseline score
df["baseline_score"] = 0

# Older content
df.loc[df["content_age_days"] >= 365, "baseline_score"] += 40

# Declining traffic
df.loc[df["trend_pct"] < -10, "baseline_score"] += 25

# Low CTR
df.loc[df["ctr"] < 2, "baseline_score"] += 20

# Good visibility
df.loc[df["impressions_90d"] >= 1000, "baseline_score"] += 15

# Action label
df["action"] = np.where(
    df["baseline_score"] >= 60,
    "Refresh Content",
    "Monitor"
)

# Reason code
df["reason_code"] = np.where(
    df["baseline_score"] >= 60,
    "REFRESH_STALE",
    "LOW_PRIORITY"
)

# Rank pages
ranked = (
    df.sort_values("baseline_score", ascending=False)
      .reset_index(drop=True)
)

# Save CSV
Path("work/outputs").mkdir(parents=True, exist_ok=True)
ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

# Show top 20
print(
    ranked[
        [
            "content_id",
            "baseline_score",
            "action",
            "reason_code"
        ]
    ].head(20)
)

              content_id  baseline_score           action    reason_code
0   content_a00c249b224d             100  Refresh Content  REFRESH_STALE
1   content_a1fb4e703a9e             100  Refresh Content  REFRESH_STALE
2   content_331d6c4de07b             100  Refresh Content  REFRESH_STALE
3   content_3d4004b21c6f             100  Refresh Content  REFRESH_STALE
4   content_7ea135180dd9             100  Refresh Content  REFRESH_STALE
5   content_78b97b24e7b9             100  Refresh Content  REFRESH_STALE
6   content_761a44afda12             100  Refresh Content  REFRESH_STALE
7   content_ab26273a7e7a             100  Refresh Content  REFRESH_STALE
8   content_48fa8051c06a             100  Refresh Content  REFRESH_STALE
9   content_d1ca928ca2da             100  Refresh Content  REFRESH_STALE
10  content_c3d1b359f295             100  Refresh Content  REFRESH_STALE
11  content_84f0c3d4751c             100  Refresh Content  REFRESH_STALE
12  content_8e1ae7310cd3             100  Refresh C

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Review

The highest ranked pages were selected because they meet multiple refresh signals, including older content, declining traffic, low CTR, and good search visibility. These pages are likely to benefit from a content refresh, but the final decision should also consider business context and content quality.

In [4]:
top20 = ranked.head(20)

for i, row in top20.iterrows():
    print(f"{i+1}. Action: {row['action']}")
    print(f"   Reason Code: {row['reason_code']}")
    print(f"   Score: {row['baseline_score']}")
    print("   Confidence: Medium")
    print("   What would make it wrong: The page may be affected by seasonal trends or recent content changes that are not included in the baseline rule.")
    print()

1. Action: Refresh Content
   Reason Code: REFRESH_STALE
   Score: 100
   Confidence: Medium
   What would make it wrong: The page may be affected by seasonal trends or recent content changes that are not included in the baseline rule.

2. Action: Refresh Content
   Reason Code: REFRESH_STALE
   Score: 100
   Confidence: Medium
   What would make it wrong: The page may be affected by seasonal trends or recent content changes that are not included in the baseline rule.

3. Action: Refresh Content
   Reason Code: REFRESH_STALE
   Score: 100
   Confidence: Medium
   What would make it wrong: The page may be affected by seasonal trends or recent content changes that are not included in the baseline rule.

4. Action: Refresh Content
   Reason Code: REFRESH_STALE
   Score: 100
   Confidence: Medium
   What would make it wrong: The page may be affected by seasonal trends or recent content changes that are not included in the baseline rule.

5. Action: Refresh Content
   Reason Code: REFRESH_S

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks

Some pages may receive a high score because they are old or have declining traffic, even if they do not actually need a content refresh. Seasonal content, niche pages, or recently updated pages may also appear in the ranked list.

### Leakage Check

The baseline rule only uses current signals such as content age, CTR, impressions, and traffic trend. No future information or target labels are used, so the rule does not introduce feature leakage.

In [5]:
print("Leakage Check Results")

future_features = False
target_labels = False

if not future_features and not target_labels:
    print("✓ No future-window features used.")
    print("✓ No target labels used.")
    print("Leakage check passed.")
else:
    print("Potential leakage detected.")

Leakage Check Results
✓ No future-window features used.
✓ No target labels used.
Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.